# Solution 07 — the Chat class

History lives on the object. `ask()` carries it between turns; `save()` and `load()` go through
`ModelMessagesTypeAdapter`. The file goes in `data/`, which git ignores.

In [ ]:
from learningdantic import setup

MODEL = setup()

In [ ]:
from pathlib import Path

from pydantic_ai import Agent, ModelMessagesTypeAdapter

tutor = Agent(
    MODEL, instructions="You are a friendly science tutor. Keep answers to two sentences."
)


class Chat:
    def __init__(self, agent: Agent) -> None:
        self.agent = agent
        self.history = []  # the messages so far

    async def ask(self, prompt: str) -> str:
        """One turn: run with the history so far, then keep the longer history."""
        result = await self.agent.run(prompt, message_history=self.history)
        self.history = result.all_messages()
        return result.output

    def save(self, path: Path) -> None:
        """Write the whole conversation to a JSON file."""
        path.write_bytes(ModelMessagesTypeAdapter.dump_json(self.history))

    @classmethod
    def load(cls, agent: Agent, path: Path) -> "Chat":
        """Build a new Chat from a saved file."""
        chat = cls(agent)
        chat.history = ModelMessagesTypeAdapter.validate_json(path.read_bytes())
        return chat

In [ ]:
Path("../data").mkdir(exist_ok=True)
path = Path("../data/07_chat.json")
path.unlink(missing_ok=True)  # start clean, so re-running the cell works

chat = Chat(tutor)
print("turn 1:", await chat.ask("I'm curious about black holes. Where do I start?"))
print("turn 2:", await chat.ask("What is the event horizon, exactly?"))
print("turn 3:", await chat.ask("What happens if I cross it?"))
chat.save(path)

revived = Chat.load(tutor, path)  # a brand-new object, filled from the file
print()
print("restored", len(revived.history), "messages")
answer = await revived.ask("In a few words: what topic did I ask about first?")
print("proof:", answer)

In [ ]:
assert path.exists(), "the chat was not saved"
assert len(chat.history) == 6, len(chat.history)  # 3 turns x (request + response)
assert ModelMessagesTypeAdapter.dump_json(chat.history) == path.read_bytes()
assert len(revived.history) == 8, len(revived.history)  # 6 loaded + 1 new turn
assert "black hole" in answer.lower(), answer
print("all checks passed")

**Notes**

- `ask()` is the only place history changes: one clear seam, easy to test.
- `ask()` is `async def` only because it awaits `agent.run(...)`. `save` and `load` are plain `def`.
- Notebook 13 keeps the same idea in a SQLite table keyed by chat id, with two plain `def`
  functions instead of a class.